<a href="https://colab.research.google.com/github/1BF24CS033/AI/blob/main/1BF24CS033_Week3_AStar_8Puzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A* Algorithm Implementation for the 8-Puzzle
This notebook contains a complete Python implementation of the A* search algorithm. The implementation is modular and allows solving the puzzle using either the **Manhattan Distance** or the **Misplaced Tiles** heuristic function.

In [4]:
import heapq
import copy

class EightPuzzleState:
    def __init__(self, board, parent=None, action=None, g=0, h=0):
        self.board = board  # 2D list (3x3)
        self.parent = parent
        self.action = action
        self.g = g  # Cost from start to current node
        self.h = h  # Heuristic cost to goal
        self.f = g + h  # Total estimated cost

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def get_neighbors(self):
        neighbors = []
        r, c = self.get_blank_pos()
        moves = {
            'Up': (r - 1, c),
            'Down': (r + 1, c),
            'Left': (r, c - 1),
            'Right': (r, c + 1)
        }
        for action, (nr, nc) in moves.items():
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = copy.deepcopy(self.board)
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(EightPuzzleState(new_board, parent=self, action=action, g=self.g + 1))
        return neighbors

    def to_tuple(self):
        return tuple(tuple(row) for row in self.board)

# --- Heuristic Functions ---

def misplaced_tiles(board, goal_board):
    """Counts the number of misplaced tiles (excluding the blank tile)."""
    misplaced = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0 and val != goal_board[r][c]:
                misplaced += 1
    return misplaced

def manhattan_distance(board, goal_board):
    """Calculates the sum of Manhattan distances of tiles from their goal positions."""
    # Create a mapping of value -> (row, col) for fast lookup in goal board
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal_board[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                gr, gc = goal_pos[val]
                distance += abs(r - gr) + abs(c - gc)
    return distance

# --- Parent A* Search Function ---

def a_star_search(start_board, goal_board, heuristic_func):
    """
    Solves the 8-puzzle using A* search with a configurable heuristic function.
    """
    start_h = heuristic_func(start_board, goal_board)
    start_node = EightPuzzleState(start_board, g=0, h=start_h)

    open_list = []
    heapq.heappush(open_list, start_node)

    closed_set = set()

    nodes_expanded = 0

    while open_list:
        current_node = heapq.heappop(open_list)
        nodes_expanded += 1

        if current_node.board == goal_board:
            # Reconstruct path
            path = []
            curr = current_node
            while curr:
                path.append(curr)
                curr = curr.parent
            return path[::-1], nodes_expanded

        state_tuple = current_node.to_tuple()
        if state_tuple in closed_set:
            continue
        closed_set.add(state_tuple)

        for neighbor in current_node.get_neighbors():
            if neighbor.to_tuple() in closed_set:
                continue
            neighbor.h = heuristic_func(neighbor.board, goal_board)
            neighbor.f = neighbor.g + neighbor.h
            heapq.heappush(open_list, neighbor)

    return None, nodes_expanded

def print_board(board):
    for row in board:
        print(" ".join(str(val) if val != 0 else "_" for val in row))
    print()

# Define the starting and goal states
start_state = [
    [1, 2, 3],
    [0, 4, 6],
    [7, 5, 8]
]

goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

print("Initial State:")
print_board(start_state)
print("Goal State:")
print_board(goal_state)

# Solve using Manhattan Distance heuristic
print("--- Solving with Manhattan Distance Heuristic ---")
path_manhattan, expanded_manhattan = a_star_search(start_state, goal_state, manhattan_distance)
if path_manhattan:
    print(f"Solvable in {len(path_manhattan) - 1} moves. Nodes expanded: {expanded_manhattan}\n")
    for step, node in enumerate(path_manhattan):
        print(f"Step {step} (Action: {node.action}):")
        print_board(node.board)
else:
    print("No solution found.")

# Solve using Misplaced Tiles heuristic
print("--- Solving with Misplaced Tiles Heuristic ---")
path_misplaced, expanded_misplaced = a_star_search(start_state, goal_state, misplaced_tiles)
if path_misplaced:
    print(f"Solvable in {len(path_misplaced) - 1} moves. Nodes expanded: {expanded_misplaced}")
else:
    print("No solution found.")

Initial State:
1 2 3
_ 4 6
7 5 8

Goal State:
1 2 3
4 5 6
7 8 _

--- Solving with Manhattan Distance Heuristic ---
Solvable in 3 moves. Nodes expanded: 4

Step 0 (Action: None):
1 2 3
_ 4 6
7 5 8

Step 1 (Action: Right):
1 2 3
4 _ 6
7 5 8

Step 2 (Action: Down):
1 2 3
4 5 6
7 _ 8

Step 3 (Action: Right):
1 2 3
4 5 6
7 8 _

--- Solving with Misplaced Tiles Heuristic ---
Solvable in 3 moves. Nodes expanded: 4
